# MorphoNet-BT — 02: Patient-Level Five-Fold Cross-Validation
## Objective and patient separation
Create five PID-disjoint outer folds, each with Train/Validation/Test. Random slice splitting risks leakage because patients have multiple slices. Patient separation takes priority over equal slice counts. No training or preprocessing is performed by this workflow.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


## Configuration and imports

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from pathlib import Path
import sys, json, platform
PROJECT_ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
assert Path(sys.prefix).name.lower() == "dl", sys.executable
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.data.kfold import (load_manifest, patient_manifest, generate_outer_folds, add_validation,
    verify_patient_separation, audit_splits, summarize_splits, assignments,
    save_splits, reload_splits, save_metadata, PARTS)
SEED = 42
N_SPLITS = 5
VALIDATION_FRACTION = 0.20
MANIFEST = PROJECT_ROOT / "results/dataset_statistics.csv"
SPLITS = PROJECT_ROOT / "splits"
OUTPUT = PROJECT_ROOT / "results/kfold"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable, platform.python_version())

## Load Phase 1 records and patient summary
Read PID/sample_id as strings. Preserve original labels 1=Meningioma, 2=Glioma, 3=Pituitary. Do not use cvind.mat.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
frame = load_manifest(MANIFEST)
patients = patient_manifest(frame)
assert len(frame) == 3064 and len(patients) == 233, "Dataset changed: review required"
display(patients.groupby("class_name").agg(patients=("PID", "size"), slices=("num_slices", "sum")))
display(patients.num_slices.describe())

## Outer folds
StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42) acts on slice rows with groups=PID and y=original_label. It approximates slice-class proportions while keeping patients intact; patient/slice counts need not be equal. Use fixed lexical PID/sample_id ordering. Do not search other seeds after viewing results.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
outer_folds = generate_outer_folds(frame, seed=SEED)
display(pd.DataFrame([{"fold": i, "remaining_patients": r.PID.nunique(), "test_patients": t.PID.nunique(),
                       "remaining_slices": len(r), "test_slices": len(t)}
                      for i, (r, t) in enumerate(outer_folds, 1)]))

## Patient-level validation holdout
Apply stratified train_test_split to one row per remaining patient, with 20% holdout and seed 42 in every fold. Expand PID assignments to all slices. Integer rounding may yield slightly above 20%. Test patients never participate in validation selection. This is outer cross-validation with an inner holdout, not nested hyperparameter search.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
splits = add_validation(outer_folds, seed=SEED, validation_fraction=VALIDATION_FRACTION)
audit, coverage = audit_splits(splits, frame)
for row in audit.itertuples(): print(f"Fold {row.fold}: {row.status}")
display(audit)
display(pd.Series(coverage))
assert coverage["unique_test_patients"] == 233
assert coverage["unique_test_samples"] == 3064

## Verify fail-fast leakage and missing-slice checks
Temporary in-memory changes leave saved/original splits untouched.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from copy import deepcopy
bad = deepcopy(splits[1])
bad["val"] = pd.concat([bad["val"], bad["train"].iloc[[0]]], ignore_index=True)
try:
    verify_patient_separation(*(bad[p] for p in PARTS), frame)
except ValueError as exc:
    assert "LEAKAGE" in str(exc)
    print("Injected PID leakage: correctly rejected")
else:
    raise AssertionError("Leakage detector failed")
bad = deepcopy(splits[1])
bad["test"] = bad["test"].iloc[1:]
try:
    verify_patient_separation(*(bad[p] for p in PARTS), frame)
except ValueError:
    print("Missing test slice: correctly rejected")
else:
    raise AssertionError("Coverage detector failed")

## Patient/slice distributions and cohort-proportion deviations
Deviations are percentage points, not relative percentages. This descriptive audit introduces no weighting or resampling.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
summary, balance = summarize_splits(splits, frame)
display(summary)
display(balance.pivot_table(index=["fold", "split", "level"], columns="class_name", values="absolute_deviation_pp"))
test_summary = summary[summary.split.eq("test")]
test_balance = balance[balance.split.eq("test")]
balance_report = {
    "test_patient_min": int(test_summary.num_patients.min()),
    "test_patient_max": int(test_summary.num_patients.max()),
    "test_patient_range": int(test_summary.num_patients.max() - test_summary.num_patients.min()),
    "test_slice_min": int(test_summary.num_slices.min()),
    "test_slice_max": int(test_summary.num_slices.max()),
    "test_slice_range": int(test_summary.num_slices.max() - test_summary.num_slices.min()),
    "test_slice_max_min_ratio": float(test_summary.num_slices.max() / test_summary.num_slices.min()),
    "largest_test_class_deviation": test_balance.loc[test_balance.absolute_deviation_pp.idxmax()].to_dict(),
    "largest_all_splits_class_deviation": balance.loc[balance.absolute_deviation_pp.idxmax()].to_dict(),
    "max_test_deviation_pp_by_level": test_balance.groupby("level").absolute_deviation_pp.max().to_dict(),
}
display(pd.Series(balance_report))

## Figures: Test class distributions and Train/Validation/Test sizes

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
class_order = ["glioma", "meningioma", "pituitary"]
for ax, unit in zip(axes[0], ["patients", "slices"]):
    test_summary.set_index("fold")[[f"{c}_{unit}" for c in class_order]].rename(
        columns={f"{c}_{unit}": c.title() for c in class_order}).plot.bar(ax=ax, rot=0)
    ax.set_title(f"Test: {unit} per class")
    ax.set_xlabel("Outer fold")
    ax.set_ylabel("Count")
for ax, unit in zip(axes[1], ["patients", "slices"]):
    summary.pivot(index="fold", columns="split", values=f"num_{unit}")[["train", "val", "test"]].plot.bar(ax=ax, rot=0)
    ax.set_title(f"Total {unit} by partition")
    ax.set_xlabel("Outer fold")
    ax.set_ylabel("Count")
fig.tight_layout()
fig.savefig(OUTPUT / "fold_distributions.png", dpi=300)
fig.savefig(OUTPUT / "fold_distributions.pdf")
plt.show()

## Split reproducibility
Recreate with the same seed and after shuffling input rows then restoring canonical order. All sample assignments must match.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
repeat = add_validation(generate_outer_folds(frame, seed=SEED), seed=SEED, validation_fraction=VALIDATION_FRACTION)
reordered = frame.sample(frac=1, random_state=123).sort_values(["PID", "sample_id"], kind="stable").reset_index(drop=True)
repeat_reordered = add_validation(generate_outer_folds(reordered, seed=SEED), seed=SEED, validation_fraction=VALIDATION_FRACTION)
for fold in splits:
    for name in PARTS:
        pd.testing.assert_frame_equal(splits[fold][name], repeat[fold][name])
        pd.testing.assert_frame_equal(splits[fold][name], repeat_reordered[fold][name])
print("Reproducibility: PASS (same seed and canonical input order)")

## Save CSVs and reproducibility metadata
Each file retains ZIP source/member, archive::member file_path, PID, sample_id, original label and other Phase 1 metadata.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
save_splits(splits, SPLITS)
patients.to_csv(SPLITS / "patient_manifest.csv", index=False)
summary.to_csv(OUTPUT / "fold_summary.csv", index=False)
balance.to_csv(OUTPUT / "class_balance.csv", index=False)
assignments(splits).to_csv(OUTPUT / "patient_fold_assignment.csv", index=False)
metadata = save_metadata(OUTPUT / "split_metadata.json", MANIFEST, SPLITS, seed=SEED)
(OUTPUT / "balance_report.json").write_text(json.dumps(balance_report, indent=2), encoding="utf-8")
print("Saved 15 split CSV files and patient manifest")
display(pd.Series({k: v for k, v in metadata.items() if k != "split_sha256"}))

## Reload saved files and audit
Repeat intersection and coverage checks, match each sample against Phase 1 records and compare with original assignments.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
saved = reload_splits(SPLITS)
disk_audit, disk_coverage = audit_splits(saved, frame)
for fold in splits:
    for name in PARTS:
        pd.testing.assert_frame_equal(saved[fold][name], splits[fold][name], check_dtype=False)
        assert len(saved[fold][name]) == len(splits[fold][name])
saved_patients = pd.read_csv(SPLITS / "patient_manifest.csv", dtype={"PID": str})
pd.testing.assert_frame_equal(saved_patients, patients)
saved_assignments = pd.read_csv(OUTPUT / "patient_fold_assignment.csv", dtype={"PID": str})
pd.testing.assert_frame_equal(saved_assignments, assignments(saved))
disk_audit.to_csv(OUTPUT / "saved_split_audit.csv", index=False)
disk_coverage.update(reproducibility="PASS", roundtrip="PASS", injected_leakage_rejected=True, missing_slice_rejected=True)
(OUTPUT / "final_audit.json").write_text(json.dumps(disk_coverage, indent=2), encoding="utf-8")
for row in disk_audit.itertuples(): print(f"Fold {row.fold}: PASS — saved CSVs; PID overlaps = 0")
display(disk_audit)
display(pd.Series(disk_coverage))

## Interpretation and historical review boundary
Each patient and slice appears exactly once in outer Test, with disjoint patients within every fold. Patients may recur in training sets across folds; this is expected and is not within-fold leakage. Variable slices per patient affect fold sizes and patient-versus-slice metrics. Recorded PID separation does not establish hospital/scanner independence or detect unknown identity errors. Library versions, input order and hashes are recorded; saved assignments are authoritative for later experiments. Test statistics must not guide weights, hyperparameters or model selection. Historically this phase stopped before Phase 3 for review.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(Markdown(f"**Execution summary:** {disk_coverage['unique_test_patients']} patients and{disk_coverage['unique_test_samples']} images, each covered once in Test. Test size: {balance_report['test_patient_min']}–{balance_report['test_patient_max']} patients and{balance_report['test_slice_min']}–{balance_report['test_slice_max']} images. Maximum Test class deviation: {test_balance.absolute_deviation_pp.max():.2f} percentage points. All saved-file checks passed."))